# Antigen-Epitope analysis of gene conversion events - Try 1

# Import Statements (libraries + Functions)

In [1]:
import numpy as np
import pandas as pd
import vcf
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

%matplotlib inline

In [2]:
from Bio import SeqIO
from Bio.Seq import Seq

In [3]:
import json

In [4]:
# https://bioframe.readthedocs.io/en/latest/guide-intervalops.html
import bioframe as bf


In [5]:
import subprocess

#### Pandas Viewing Settings

In [6]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# Define sets of coord cols for using BioFrame

In [7]:
Query_CoordCols = ("Query_Name", "Query_Start", "Query_End")
HmReg_CoordCols = ("Chr", "Start", "End")
HmRegion_CoordCols = HmReg_CoordCols
Epitope_CoordCols = ("Chrom", "Rv_Start", "Rv_End")
RE_CoordCols = ("seqname", "start_0based", "end_1based")
GenomeAnno_CoordCols = ("Chrom", "Start", "End")


## Define function for parsing XLSX file from IEDB

In [8]:
def parse_IEDB_peptides_XLSX(IEDB_excel_file, sheet_name='Sheet1'):
    # Load the Excel file
    all_excel_data = pd.read_excel(IEDB_excel_file, sheet_name=sheet_name)

    # Define columns to select
    columns_to_select = [
        "Epitope - Name", 
        "Epitope - Molecule Parent",
        "Epitope - Source Molecule", 
        "Epitope ID - IEDB IRI", 
        "Epitope - Molecule Parent IRI"
    ]

    # Select and rename columns
    peptides_df = all_excel_data[columns_to_select]
    peptides_df.columns = [
        "Epitope_Seq", 
        "Molecule_Parent", 
        "Source_Molecule", 
        "IEDB_ID_URL", 
        "UniProt_ID_URL"
    ]

    # Calculate the AA length and extract IDs
    peptides_df["AA_Length"] = peptides_df["Epitope_Seq"].str.len()
    peptides_df["Epitope_ID"] = peptides_df["IEDB_ID_URL"].str.split("/").str[-1]
    peptides_df["UniProt_ID"] = peptides_df["UniProt_ID_URL"].str.split("/").str[-1]

    # Reorder columns
    new_col_order = [
        'Epitope_ID', 
        'Epitope_Seq', 
        'AA_Length', 
        'Molecule_Parent', 
        'Source_Molecule', 
        "UniProt_ID"
    ]

    peptides_df = peptides_df[new_col_order]

    return peptides_df

# Example usage
# result_df = parse_IEDB_peptides_XLSX(Panda2024_IEDB_AllAssayedPeptides_XLSX)
# print(result_df.shape)

### Define function for matching epitope/peptide sequences to H37Rv reference proteins

In [9]:

def map_epitopes_to_proteins(in_Epitope_DF, dictOf_H37Rv_ProtSeq, RvID_To_Symbol_Dict, Symbol_To_RvID_Dict, H37Rv_GenomeAnno_Genes_DF):
    listOf_EpiRowTuples = []

    for i, row in tqdm(in_Epitope_DF.iterrows()):
        # a) Pull out all epitope info from the row
        i_Epi_ID = row["Epitope_ID"]
        i_Epitope_Seq = row["Epitope_Seq"]
        i_Epitope_Len = len(i_Epitope_Seq)

        listOf_AllProtein_Wi_Epitope = []

        for prot_RvID, AAseq in dictOf_H37Rv_ProtSeq.items():
            try:
                prot_Symbol = RvID_To_Symbol_Dict[prot_RvID]
            except KeyError:
                prot_Symbol = prot_RvID

            i_Count_OvrLap = AAseq.count_overlap(i_Epitope_Seq)

            if i_Count_OvrLap > 0:
                 
                #i_RvID_WiEpi = Symbol_To_RvID_Dict[prot_Symbol]
                i_Protein_Complete_AASeq = dictOf_H37Rv_ProtSeq[prot_RvID]

                EpiPos_Start_Find_In_AAseq = i_Protein_Complete_AASeq.find(i_Epitope_Seq)
                EpiPos_End_Find_In_AAseq = EpiPos_Start_Find_In_AAseq + i_Epitope_Len

                i_Epitope_AA_Start_0based = EpiPos_Start_Find_In_AAseq
                i_Epitope_AA_End = EpiPos_End_Find_In_AAseq

                # Get genomic coordinates
                i_Gene_Info = H37Rv_GenomeAnno_Genes_DF.query(f"Symbol == '{prot_Symbol}'").head(1)
                if i_Gene_Info.shape[0] > 0:
                    Gene_Genomic_Start_0based = i_Gene_Info["Start"].values[0]
                    Gene_Genomic_End = i_Gene_Info["End"].values[0]
                    Gene_Genomic_Strand = i_Gene_Info["Strand"].values[0]
    
                    if Gene_Genomic_Strand == "+":
                        i_Epi_Rv_Start = Gene_Genomic_Start_0based + (i_Epitope_AA_Start_0based * 3)
                        i_Epi_Rv_End = Gene_Genomic_Start_0based + (i_Epitope_AA_End * 3)
                    elif Gene_Genomic_Strand == "-":
                        i_Epi_Rv_End = Gene_Genomic_End - (i_Epitope_AA_Start_0based * 3)
                        i_Epi_Rv_Start = Gene_Genomic_End - (i_Epitope_AA_End * 3)
                    else:
                        print(i_Epi_ID, "Strand is not + or - !!!!")
    
                    Chrom = "NC_000962.3"
                    i_Row_Tuple = (i_Epi_ID, i_Epitope_Seq, i_Epitope_Len,
                                   prot_RvID, prot_Symbol,
                                   EpiPos_Start_Find_In_AAseq, EpiPos_End_Find_In_AAseq,
                                   Chrom, i_Epi_Rv_Start, i_Epi_Rv_End, i_Count_OvrLap)
    
                    listOf_EpiRowTuples.append(i_Row_Tuple)

    i_Epitopes_Mapped_QC_DF = pd.DataFrame(listOf_EpiRowTuples)

    i_Epitopes_Mapped_QC_DF.columns = ["Epitope_ID", "Epitope_Seq", "Epitope_Len",
                                            "RvID", "Symbol",
                                            "AA_Start", "AA_End",
                                            "Chrom", "Rv_Start", "Rv_End", "EpitopeSeqFreqInAntigen"]
    
    return i_Epitopes_Mapped_QC_DF


# H37rv genome anno parsing

In [10]:
RepoRef_Dir = "../../References"
ESX_Genes_List_TSV = f"{RepoRef_Dir}/190927_H37rv_ListOf_ESXgenes.tsv"
AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir = f"{RepoRef_Dir}/201027_H37rv_AnnotatedGenes_And_IntergenicRegions"

H37Rv_GenomeAnnotations_Genes_V2_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.V2.tsv"

## H37Rv Gene Annotations TSV
H37Rv_GenomeAnno_Genes_DF = pd.read_csv(H37Rv_GenomeAnnotations_Genes_V2_TSV, sep = "\t")

RvID_To_Symbol_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['H37rv_GeneID', 'Symbol']].values)
Symbol_To_RvID_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['Symbol', 'H37rv_GeneID']].values)
Symbol_To_FuncCat_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['Symbol', 'Functional_Category']].values)

Esx_Genes_DF = pd.read_csv(ESX_Genes_List_TSV, sep = '\t')
ListOf_Esx_Symbols = list(Esx_Genes_DF["symbol"].values)
ListOf_Esx_RvIDs = list(Esx_Genes_DF["gene_id"].values)

listOf_PEPPE_Symbols = list( H37Rv_GenomeAnno_Genes_DF.query(" Functional_Category == 'PE/PPE' ")["Symbol"].values )
listOf_PEPPE_RvIDs = list( H37Rv_GenomeAnno_Genes_DF.query(" Functional_Category == 'PE/PPE' ")["H37rv_GeneID"].values )

listOf_13E12_Region_RvIDs = ["Rv0094c", "Rv0095c", "Rv0393", "Rv1572c", "Rv1572c", "Rv1128c", "Rv1148c", "Rv1587c", "Rv1588c", "Rv1702c", "Rv1945", "Rv2100", "Rv3466", "Rv3467"]


In [11]:
H37Rv_GenomeAnno_Genes_DF.head(2)

,Chrom,Start,End,Strand,H37rv_GeneID,Symbol,Feature,Functional_Category,Is_Pseudogene,Product,PEandPPE_Subfamily,ExcludedGroup_Category,Gene_Cat_V2
0,NC_000962.3,0,1524,+,Rv0001,dnaA,CDS,information pathways,No,Chromosomal replication initiator protein DnaA,NaN,NotExcluded,information pathways
1,NC_000962.3,2051,3260,+,Rv0002,dnaN,CDS,information pathways,No,DNA polymerase III (beta chain) DnaN (DNA nucl...,NaN,NotExcluded,information pathways


# Parse H37Rv genome sequence

In [12]:
H37rv_Ref_GBK_PATH = "/n/data1/hms/dbmi/farhat/mm774/References/GCF_000195955.2_ASM19595v2_genomic.gbk"
H37Rv_FA = "/n/data1/hms/dbmi/farhat/mm774/References/GCF_000195955.2_ASM19595v2_genomic.fasta"

H37Rv_Seq = SeqIO.read(H37Rv_FA, "fasta").seq
len(H37Rv_Seq)

4411532

# Parse H37Rv Protein AA sequences

In [13]:
O2_RefDir = "/n/data1/hms/dbmi/farhat/mm774/References"

MycoBrowser_RefFiles_Dir = f"{O2_RefDir}/190619_Mycobrowser_H37rv_ReferenceFiles"

H37Rv_Proteins_MycoBro_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta"
H37Rv_Proteins_NCBI_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta"

H37Rv_FAA_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_proteins.faa"
H37Rv_GBK_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_genomic.gbk"

In [14]:
#!ls -1 $MycoBrowser_RefFiles_Dir

#### Parse MycoBrowser Protein Seq Ref

In [15]:
dictOf_H37Rv_MycoBrow_ProtSeq = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37Rv_Proteins_MycoBro_FAA, "fasta"))):
    ShortID = record.name
    
    dictOf_H37Rv_MycoBrow_ProtSeq[ShortID] = record.seq


4090it [00:00, 130501.65it/s]


#### Parse NCBI Protein Seq Ref

In [16]:
dictOf_H37Rv_ProtSeq_NCBI = {}
dictOf_H37Rv_ProtRecord_NCBI = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37Rv_FAA_PATH, "fasta"))):
    Rec_Description = record.description
    dict_Attr = {}
    for i in Rec_Description.split(" "):
        ###Just looking for line with " " character (as key = value)
        if "=" in i:
            key = i.strip().split("=")[0].strip('"').strip('[')
            value = i.strip().split("=")[1].strip('"').strip(']')
            ###Put them in a dictionnary
            dict_Attr[key]=value
    
    ShortID = dict_Attr["locus_tag"]
    dictOf_H37Rv_ProtSeq_NCBI[ShortID] = record.seq
    dictOf_H37Rv_ProtRecord_NCBI[ShortID] = record

# Manually define the esxM translation (Truncated in H37Rv)
dictOf_H37Rv_ProtSeq_NCBI["Rv1792"] = Seq("MASRFMTDPHAMRDMAGRFEVHAQTVEDEARRMWASAQNISGAGWSGMAEATSLDTMTMNQAFRNIVNMLHGVRDGLVRDANNYEQQEQASQQILSS")



3906it [00:00, 85574.79it/s]


In [17]:
dictOf_H37Rv_ProtSeq_NCBI["Rv1196"]

Seq('MVDFGALPPEINSARMYAGPGSASLVAAAQMWDSVASDLFSAASAFQSVVWGLT...AAG')

In [18]:
dictOf_H37Rv_ProtSeq_NCBI["Rv1792"]

Seq('MASRFMTDPHAMRDMAGRFEVHAQTVEDEARRMWASAQNISGAGWSGMAEATSL...LSS')

# Parse in H37Rv homology map

In [19]:
Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V8"
H37_Rv_MM2_HomologyMapping_Dir = f"{Main_Project_Dir}/220502.H37Rv.HomologyMapping.k19w19.ProcessedData"       

# Define paths to output TSVS

### Homologous regions (MERGED)
H37Rv_HomologyRegions_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologousRegions.k19w19.tsv"

### Homology map (pairwise alignments)
H37Rv_HomologyMap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.tsv"
H37Rv_HomologyMap_NoOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.NoOverlap.tsv"
H37Rv_HomologyMap_OnlyOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.OnlyOverlap.tsv"

H37Rv_HomologyMap_FiltAndAnno_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HmMap.k19w19.NoOverlap.Processed.V2.tsv"

### Parse in labeled homology-regions (w/ unique IDs)
HM_MergedRegions_Anno_DF = pd.read_csv(H37Rv_HomologyRegions_TSV, sep="\t")


### Variants from the homology map alignments
H37Rv_HmMap_Var_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.variants.tsv"
H37Rv_HmMap_Var_SNPs_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.variants.snps.tsv"

### Parse in homology-map DFs (pairwise alignments between all homologous regions)
Mtb_HM_PAF_DF = pd.read_csv(H37Rv_HomologyMap_TSV, sep="\t")
Mtb_HM_PAF_DF_NoOverlapRegions = pd.read_csv(H37Rv_HomologyMap_NoOverlap_TSV, sep="\t")
Mtb_HM_PAF_DF_OnlyOverlapRegions = pd.read_csv(H37Rv_HomologyMap_OnlyOverlap_TSV, sep="\t")

HmPair_DF = pd.read_csv(H37Rv_HomologyMap_FiltAndAnno_TSV, sep="\t")  

## Parse in HomologyMap variants DFs
Mtb_HM_Var_DF = pd.read_csv(H37Rv_HmMap_Var_TSV, sep="\t")
Mtb_HM_Var_SNPs_DF = pd.read_csv(H37Rv_HmMap_Var_SNPs_TSV, sep="\t")

TarCol = ['Query_Name', 'Query_Start', 'Query_End', 'Ref', 'Alt', 'SNP'] 
HM_Var_SNPs_Trim_DF = Mtb_HM_Var_SNPs_DF[TarCol]
HM_Var_SNPs_TrimUnq_DF = HM_Var_SNPs_Trim_DF.drop_duplicates()

# Gameplan for analysis:

1) Organize existing epitope data from 3 sources
   - epitope data from Scriba paper & Panda-2024 paper
        - Panda Paper Link: https://www.nature.com/articles/s41467-024-45058-9
          
3) Parse assayed peptides (Positive & Negative Epitopes) from the Panda24 and Lindestam16 studies
4) Unify the processing of peptides from all sources
   


# Define file paths for all IEDB Epitope/Peptide info from two datasets (`Panda2024` and `Lind2016`)

In [20]:
Repo_Epitope_MainDir = "../../Data/220813_MtbEpitopes"

### Lindestam-2016 Data
Lindestam2016_Epitope_Dir = f"{Repo_Epitope_MainDir}/Lindestam2016.PlosPatho.SuppData.Epitopes"

Lindestam2016_Supp_XLSV = f"{Lindestam2016_Epitope_Dir}/Lindestam2016.SuppTablesMerged.xlsx"
Lindestam2016_IEDB_AllPeptides_XLSV = f"{Lindestam2016_Epitope_Dir}/IEDB.Lindestam2016.AllAssays.tcell_table.ID_1031648.xlsx"


### Panda2024 (IEDB Submission) Positive and Negative Epitopes

Panda2024_Epitope_Dir = f"{Repo_Epitope_MainDir}/Panda2024_EpitopeMapping"

Panda2024_IEDB_AllPosEpitopes_XLSX = f"{Panda2024_Epitope_Dir}/Panda2024.IEDB.SubID_1000914.epitopes.xlsx"
Panda2024_IEDB_AllAssayedPeptides_XLSX = f"{Panda2024_Epitope_Dir}/IEDB.1000914.Panda2024.AllPeptidesAssayed.xlsx"


## A) Parse & process Lindestam-2016 Supplemental Data + Epitopes'

Submission ID on IEDB: 1031648

#### Notes on contents of Supplemental Tables from Lindestam-2016

- Table S1. The most commonly recognized 37 epitopes defined in TB Vaccine and IGRA antigens <br>
- Table S2. The most commonly recognized 38 epitopes defined from previously described epitopes. <br>
- Table S3. HLA type of adult donor cohort <br>
- Table S4. HLA restriction and penetrance of Mtb epitopes <br>
- Table S5. Peptides in each megapool <br>

In [21]:
# Parse the sheets "TableS1" and "TableS2"
L2016_S1_DF = pd.read_excel(Lindestam2016_Supp_XLSV, sheet_name="TableS1")
L2016_S2_DF = pd.read_excel(Lindestam2016_Supp_XLSV, sheet_name="TableS2")
L2016_S3_DF = pd.read_excel(Lindestam2016_Supp_XLSV, sheet_name="TableS3")
L2016_S4_DF = pd.read_excel(Lindestam2016_Supp_XLSV, sheet_name="TableS4")
L2016_S5_DF = pd.read_excel(Lindestam2016_Supp_XLSV, sheet_name="TableS5")


### Preview the Lindestam-2016 Tables (S1-S5)

#### Lindestam-2016 - Table S1 Preview

In [22]:
L2016_S1_DF.head()

,Category,Antigen,Epitope sequence,"Responding donors, n",Total magnitude of response (SFC)
0,Cell wall and cell processes,Rv0288,MSQIMYNYPAMLGHA,12,2461
1,Cell wall and cell processes,Rv0288,SAWQGDTGITYQAWQ,10,1215
2,Cell wall and cell processes,Rv0288,HEANTMAMMARDTAE,10,920
3,Cell wall and cell processes,Rv0288,YAGTLQSLGAEIAVE,5,958
4,Cell wall and cell processes,Rv0288,VRAYHAMSSTHEANT,5,868


#### Lindestam-2016 - Table S2 Preview

In [23]:
L2016_S2_DF.head()

,Antigen,Epitope sequence,"Responding donors, n",Total magnitude of response (SFC),Reactivity in both sets
0,Rv0124,AQIYQAVSAQAAAIH,2,73,NaN
1,Rv0129c,PSPSMGRDIKVQFQS,3,140,NaN
2,Rv0256c,AVLVATNFFGINTIP,6,455,NaN
3,Rv0280,GINTIPIAINEAEYV,7,1346,NaN
4,Rv0288,AAFQGAHARFVAAAA,12,1355,NaN


#### Lindestam-2016 - Table S3 Preview

In [24]:
L2016_S3_DF.head()

,Donor ID,DRB1,DRB1.1,DRB3/4/5,DRB3/4/5.1,DQA1,DQA1.1,DQB1,DQB1.1,DPA1,DPA1.1,DPB1,DPB1.1
0,TCA-001,DRB1*12:01,DRB1*15:03,DRB3*01:01,DRB5*01:01,DQA1*01:01,DQA1*01:02,DQB1*05:01,DQB1*06:02,DPA1*02:01,DPA1*02:02,DPB1*01:01,DPB1*13:01
1,TCA-002,DRB1*13:01,DRB1*15:01,DRB3*02:02,DRB3*02:02,DQA1*01:02,DQA1*01:03,DQB1*06:02,DQB1*06:04,DPA1*01:03,DPA1*02:01,DPB1*04:01,DPB1*11:01
2,TCA-003,DRB1*03:01,DRB1*07:01,DRB3*02:02,DRB4*01:03,DQA1*02:01,DQA1*05:01,DQB1*02:01,DQB1*05:03,DPA1*01:03,DPA1*02:01,DPB1*13:01,DPB1*17:01
3,TCA-004,DRB1*11:01,DRB1*15:03,DRB3*02:02,DRB5*01:01,DQA1*01:02,DQA1*05:05,DQB1*06:02,DQB1*06:02,DPA1*01:03,DPA1*03:01,DPB1*03:01,DPB1*105:01
4,TCA-005,DRB1*01:01,DRB1*07:01,DRB3*02:02,DRB4*01:03,DQA1*02:01,DQA1*05:01,DQB1*03:02,DQB1*03:02,DPA1*01:03,DPA1*02:01,DPB1*01:01,DPB1*17:01


#### Lindestam-2016 - Table S4 Preview

In [25]:
L2016_S4_DF.head()

,Antigen,Epitope sequence,Restricting HLA allele,"Donors with allele-restricted response, n","Donors with HLA allele tested, n",Penetrance (%)
0,Rv3874,AQAAVVRFQEAANKQ,DRB5*01:01,6,7,0.857143
1,Rv3874,QAAVVRFQEAANKQK,DRB5*01:01,6,10,0.600000
2,Rv3619c,DAHGAMIRAQAGSLE,DQB1*06:02,5,5,1.000000
3,Rv3874,ISTNIRQAGVQYSRA,DRB3*02:02,5,7,0.714286
4,Rv3874,EISTNIRQAGVQYSR,DRB3*02:02,5,8,0.625000


#### Lindestam-2016 - Table S5 Preview

In [26]:
L2016_S5_DF.head()

,Protein category,Antigen,MTB300,MTB125,MTB66
0,PE/PPE,"Rv0124, Rv0297, Rv1243c, Rv1788, Rv1791, Rv2490c",AQIYQAVSAQAAAIH,AQIYQAVSAQAAAIH,AQIYQAVSAQAAAIH
1,intermediary metabolism and respiration,Rv0125,VAQVGPQVVNINTKL,VAQVGPQVVNINTKL,VAQVGPQVVNINTKL
2,Lipid metabolism,"Rv0129c, Rv1886c, Rv3804c",PSPSMGRDIKVQFQS,NaN,PSPSMGRDIKVQFQS
3,PE/PPE,"Rv0256c, Rv0280, Rv0286, Rv0453, Rv1387, Rv301...",AVLVATNFFGINTIP,AVLVATNFFGINTIP,AVLVATNFFGINTIP
4,PE/PPE,"Rv0280, Rv0286, Rv0453, Rv1387, Rv3018c, Rv3021c",GINTIPIAINEAEYV,GINTIPIAINEAEYV,GINTIPIAINEAEYV


## Merge all positive epitopes from Lindestam-2016 (Tables S1, S2, S4)

#### Merge all unique positive epitope seqs

In [27]:

L2016_S1_DF["Epitope_Seq"] = L2016_S1_DF["Epitope sequence"]
L2016_S2_DF["Epitope_Seq"] = L2016_S2_DF["Epitope sequence"]
L2016_S4_DF["Epitope_Seq"] = L2016_S4_DF["Epitope sequence"]

# Get unique epitope sequences from Tables S1, S2, and S4
S1_Unq_EpiSeqs = set(L2016_S1_DF["Epitope_Seq"].unique())
S2_Unq_EpiSeqs = set(L2016_S2_DF["Epitope_Seq"].unique())
S4_Unq_EpiSeqs = set(L2016_S4_DF["Epitope_Seq"].unique())
S5_MTB300_Unq_EpiSeqs = set(L2016_S5_DF["MTB300"].unique())

# Combine all unique sequences
All_Lindestam_EpitopeSeqs = list(S1_Unq_EpiSeqs.union(S2_Unq_EpiSeqs).union(S4_Unq_EpiSeqs).union(S5_MTB300_Unq_EpiSeqs))
len(All_Lindestam_EpitopeSeqs)

320

#### Create DF for all unique positive epitope seqs

In [28]:
# Concat all epitopes from ST 1,2,4
L2016_All_PosEpitopes_DF = pd.DataFrame(pd.Series(All_Lindestam_EpitopeSeqs))
print(L2016_All_PosEpitopes_DF.shape)

L2016_All_PosEpitopes_DF = L2016_All_PosEpitopes_DF.drop_duplicates()
print(L2016_All_PosEpitopes_DF.shape)

L2016_All_PosEpitopes_DF.columns = ["Epitope_Seq"]

L2016_All_PosEpitope_Seqs = list(L2016_All_PosEpitopes_DF["Epitope_Seq"].values)

# Give each epitope a unique ID
L2016_All_PosEpitopes_DF["Epitope_ID"] = "Lind2016_Epitope_" + (L2016_All_PosEpitopes_DF.index + 1).astype(str)
L2016_All_PosEpitopes_DF.shape

print(L2016_All_PosEpitopes_DF.shape)


(320, 1)
(320, 1)
(320, 2)


In [29]:
L2016_All_PosEpitopes_DF.shape

(320, 2)

#### Peak at overlap between Lindestam2016 tables

```
Here's the summary of the overlap of unique sequences in Tables S1, S2, and S4:

Table S1 has 37 unique epitope sequences.
Table S2 has 38 unique epitope sequences.
Table S4 has 116 unique epitope sequences.
Regarding overlaps:

There are no sequences common between Tables S1 and S2.
There are 26 sequences common between Tables S1 and S4.
There are 24 sequences common between Tables S2 and S4.
No sequences are common across all three tables (S1, S2, and S4). ​
```

In [30]:

# Find overlaps between the unique sequences in S1, S2, and S4
Overlap_S1_S2 = S1_Unq_EpiSeqs.intersection(S2_Unq_EpiSeqs)
Overlap_S1_S4 = S1_Unq_EpiSeqs.intersection(S4_Unq_EpiSeqs)
Overlap_S2_S4 = S2_Unq_EpiSeqs.intersection(S4_Unq_EpiSeqs)

# Find sequences common to all three tables
Common_All = S1_Unq_EpiSeqs.intersection(S2_Unq_EpiSeqs).intersection(S4_Unq_EpiSeqs)

len(S1_Unq_EpiSeqs), len(S2_Unq_EpiSeqs), len(S4_Unq_EpiSeqs), len(Overlap_S1_S2), len(Overlap_S1_S4), len(Overlap_S2_S4), len(Common_All)


(37, 38, 116, 0, 26, 24, 0)

In [31]:
L2016_S5_DF.head(3)

,Protein category,Antigen,MTB300,MTB125,MTB66
0,PE/PPE,"Rv0124, Rv0297, Rv1243c, Rv1788, Rv1791, Rv2490c",AQIYQAVSAQAAAIH,AQIYQAVSAQAAAIH,AQIYQAVSAQAAAIH
1,intermediary metabolism and respiration,Rv0125,VAQVGPQVVNINTKL,VAQVGPQVVNINTKL,VAQVGPQVVNINTKL
2,Lipid metabolism,"Rv0129c, Rv1886c, Rv3804c",PSPSMGRDIKVQFQS,NaN,PSPSMGRDIKVQFQS


# Parse IEDB Results - All assayed peptides - `Lind2016`

Submission ID on IEDB: 1031648

### `Lind-2016` - parse all ASSAYED peptides (N = 1025)

In [32]:
# Load the main sheet 'Sheet1' to explore the data
L16_IEDB_All_IEDB_Sheet = pd.read_excel(Lindestam2016_IEDB_AllPeptides_XLSV, sheet_name='Sheet1')

# Display the first few rows of the dataframe and the columns
L16_IEDB_All_IEDB_Sheet.head(1)

,Assay ID - IEDB IRI,Reference - IEDB IRI,Reference - Type,Reference - PMID,Reference - Submission ID,Reference - Authors,Reference - Journal,Reference - Date,Reference - Title,Epitope - IEDB IRI,Epitope - Object Type,Epitope - Name,Epitope - Reference Name,Epitope - Modified residues,Epitope - Modifications,Epitope - Starting Position,Epitope - Ending Position,Epitope - IRI,Epitope - Synonyms,Epitope - Source Molecule,Epitope - Source Molecule IRI,Epitope - Molecule Parent,Epitope - Molecule Parent IRI,Epitope - Source Organism,Epitope - Source Organism IRI,Epitope - Species,Epitope - Species IRI,Epitope - Epitope Comments,Related Object - Epitope Relation,Related Object - Object Type,Related Object - Name,Related Object - Starting Position,Related Object - Ending Position,Related Object - IRI,Related Object - Synonyms,Related Object - Source Molecule,Related Object - Source Molecule IRI,Related Object - Molecule Parent,Related Object - Molecule Parent IRI,Related Object - Source Organism,Related Object - Source Organism IRI,Related Object - Species,Related Object - Species IRI,Host - Name,Host - IRI,Host - Geolocation,Host - Geolocation IRI,Host - Sex,Host - Age,Host - MHC Present,1st in vivo Process - Process Type,1st in vivo Process - Disease,1st in vivo Process - Disease IRI,1st in vivo Process - Disease Stage,1st immunogen - Epitope Relation,1st immunogen - Object Type,1st immunogen - Name,1st immunogen - Reference Name,1st immunogen - Starting Position,1st immunogen - Ending Position,1st immunogen - IRI,1st immunogen - Source Molecule,1st immunogen - Source Molecule IRI,1st immunogen - Molecule Parent,1st immunogen - Molecule Parent IRI,1st immunogen - Source Organism,1st immunogen - Source Organism IRI,1st immunogen - Species,1st immunogen - Species IRI,1st immunogen - Adjuvants,1st immunogen - Route,1st immunogen - Dose Schedule,2nd in vivo Process - Process Type,2nd in vivo Process - Disease,2nd in vivo Process - Disease IRI,2nd in vivo Process - Disease Stage,2nd immunogen - Epitope Relation,2nd immunogen - Object Type,2nd immunogen - Name,2nd immunogen - Reference Name,2nd immunogen - Starting Position,2nd immunogen - Ending Position,2nd immunogen - IRI,2nd immunogen - Source Molecule,2nd immunogen - Source Molecule IRI,2nd immunogen - Molecule Parent,2nd immunogen - Molecule Parent IRI,2nd immunogen - Source Organism,2nd immunogen - Source Organism IRI,2nd immunogen - Species,2nd immunogen - Species IRI,2nd immunogen - Adjuvants,2nd immunogen - Route,2nd immunogen - Dose Schedule,In vitro Process - Process Type,in vitro Responder Cell - Name,in vitro Responder Cell - IRI,in vitro Stimulator Cell - Name,in vitro Stimulator Cell - IRI,in vitro immunogen - Epitope Relation,in vitro immunogen - Object Type,in vitro immunogen - Name,in vitro immunogen - Reference Name,in vitro immunogen - Starting Position,in vitro immunogen - Ending Position,in vitro immunogen - IRI,in vitro immunogen - Source Molecule,in vitro immunogen - Source Molecule IRI,in vitro immunogen - Molecule Parent,in vitro immunogen - Molecule Parent IRI,in vitro immunogen - Source Organism,in vitro immunogen - Source Organism IRI,in vitro immunogen - Species,in vitro immunogen - Species IRI,Adoptive Transfer - Flag,Adoptive Transfer - Comments,Immunization - Comments,Assay - Location of Assay Data in Reference,Assay - Method,Assay - Response measured,Assay - Units,Assay - IRI,Assay - Qualitative Measurement,Assay - Measurement Inequality,Assay - Quantitative measurement,Assay - Number of Subjects Tested,Assay - Number of Subjects Positive,Assay - Response Frequency (%),Assay - Comments,Effector Cell - Source Tissue,Effector Cell - Source Tissue IRI,Effector Cell - Name,Effector Cell - IRI,Effector Cell - Culture Condition,Effector Cell - TCR Name,Complex - PDB ID,Antigen Presenting Cell - Source Tissue,Antigen Presenting Cell - Source Tissue IRI,Antigen Presenting Cell - Name,Antigen Presenting Cell - IRI,Antigen Presenting Cell - Culture Condition,MHC

In [33]:
# Selecting the most relevant columns to summarize
summary_columns = ['Epitope - Name',
                   'Assay - Qualitative Measurement',
                   'MHC Restriction - Name',
                   'Assay - Number of Subjects Tested',
                   'Assay - Number of Subjects Positive']

# Create a summary DataFrame with the selected columns
Lind16_IEDB_AllPeptides_DF = L16_IEDB_All_IEDB_Sheet[summary_columns] #.dropna()

Lind16_IEDB_AllPeptides_DF.rename(columns={"Epitope - Name": "Epitope_Seq"}, inplace=True)
Lind16_IEDB_AllPeptides_DF.rename(columns={"Assay - Qualitative Measurement": "Epitope_Status"}, inplace=True)
Lind16_IEDB_AllPeptides_DF.rename(columns={"MHC Restriction - Name": "HLA_Allele"}, inplace=True)
Lind16_IEDB_AllPeptides_DF.rename(columns={'Assay - Number of Subjects Tested': "Num_Assayed"}, inplace=True)
Lind16_IEDB_AllPeptides_DF.rename(columns={'Assay - Number of Subjects Positive': "Num_Positive"}, inplace=True)

Lind16_IEDB_AllPeptides_DF["HLA_Allele"].fillna("None")
Lind16_IEDB_AllPeptides_DF["Fraction_PositiveAssay"] = Lind16_IEDB_AllPeptides_DF["Num_Positive"] / Lind16_IEDB_AllPeptides_DF["Num_Assayed"]

Lind16_IEDB_AllPeptides_HLAInfo_DF = Lind16_IEDB_AllPeptides_DF.sort_values(["Fraction_PositiveAssay", "Num_Positive"], ascending = False)
Lind16_IEDB_AllPeptides_DF.shape


/tmp/ipykernel_3702272/366074419.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Lind16_IEDB_AllPeptides_DF.rename(columns={"Epitope - Name": "Epitope_Seq"}, inplace=True)
/tmp/ipykernel_3702272/366074419.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Lind16_IEDB_AllPeptides_DF.rename(columns={"Assay - Qualitative Measurement": "Epitope_Status"}, inplace=True)
/tmp/ipykernel_3702272/366074419.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-

(1025, 6)

In [34]:
Lind16_IEDB_AllPeptides_DF.head(5)

,Epitope_Seq,Epitope_Status,HLA_Allele,Num_Assayed,Num_Positive,Fraction_PositiveAssay
0,AAAQASAAAAAYEAA,Negative,NaN,63,0,0.0
1,AAASAWDGLAEELHA,Negative,NaN,63,0,0.0
2,AAASWDALAAELASA,Negative,NaN,63,0,0.0
3,AAATATATATLLPFE,Negative,NaN,63,0,0.0
4,AADMWGPSSDPAWER,Negative,NaN,63,0,0.0


In [35]:
Lind16_IEDB_AllPeptides_DF.sort_values(["Fraction_PositiveAssay", "Num_Positive"], ascending = False).head(5)

,Epitope_Seq,Epitope_Status,HLA_Allele,Num_Assayed,Num_Positive,Fraction_PositiveAssay
200,DAHGAMIRAQAGSLE,Positive,HLA-DQB1*06:02,5,5,1.0
501,ISTNIRQAGVQYSRA,Positive,HLA-DQB1*06:02,4,4,1.0
620,MHVSFVMAYPEMLAA,Positive,HLA-DQB1*06:02,4,4,1.0
106,ALPPEINSARMYAGP,Positive,HLA-DQB1*06:02,3,3,1.0
632,MSFVTTQPEALAAAA,Positive,HLA-DRB1*07:01,3,3,1.0


#### Deduplicate and trim DF

In [36]:
Lind16_AllPeptides_Trim_DF = Lind16_IEDB_AllPeptides_DF[["Epitope_Seq", "Epitope_Status"]].drop_duplicates()

# Give each peptide a unique ID
Lind16_AllPeptides_Trim_DF["Epitope_ID"] = "Lind2016_Peptide_" + (Lind16_AllPeptides_Trim_DF.index + 1).astype(str)

Lind16_IEDB_AllPeptides_SeqList = list(Lind16_AllPeptides_Trim_DF["Epitope_Seq"].unique())

Lind16_IEDB_NegPeptides_SeqList = list(Lind16_AllPeptides_Trim_DF.query("Epitope_Status == 'Negative' ")["Epitope_Seq"].unique())
Lind16_IEDB_PosEpitopes_SeqList = list(Lind16_AllPeptides_Trim_DF.query("Epitope_Status == 'Positive' ")["Epitope_Seq"].unique())

Lind16_AllPeptides_Trim_DF.shape

(761, 3)

In [37]:
Lind16_AllPeptides_Trim_DF.head(4)

,Epitope_Seq,Epitope_Status,Epitope_ID
0,AAAQASAAAAAYEAA,Negative,Lind2016_Peptide_1
1,AAASAWDGLAEELHA,Negative,Lind2016_Peptide_2
2,AAASWDALAAELASA,Negative,Lind2016_Peptide_3
3,AAATATATATLLPFE,Negative,Lind2016_Peptide_4


In [38]:
Lind16_AllPeptides_Trim_DF["Epitope_Seq"].value_counts().head(4)

Epitope_Seq
YYQSGLSIVMPVGGQ    1
AAAQASAAAAAYEAA    1
AAASAWDGLAEELHA    1
AAASWDALAAELASA    1
Name: count, dtype: int64

In [39]:
Lind16_AllPeptides_Trim_DF["Epitope_Status"].value_counts()

Epitope_Status
Negative    526
Positive    235
Name: count, dtype: int64

# Parse IEDB Results - All assayed peptides - `Panda2024`

Submission ID on IEDB: 1000914

#### `Panda-2024` - Parse all POSITIVE EPITOPES (Reactivite peptides, N = 174)

In [40]:
Panda24_IEDB_Pos_Epitope_DF = parse_IEDB_peptides_XLSX(Panda2024_IEDB_AllPosEpitopes_XLSX)
Panda24_All_PosEpitope_Seqs = list(Panda24_IEDB_Pos_Epitope_DF["Epitope_Seq"].values)

Panda24_IEDB_PosEpitopes_SeqList = Panda24_All_PosEpitope_Seqs

Panda24_IEDB_Pos_Epitope_DF.shape

/tmp/ipykernel_3702272/1950331011.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  peptides_df["AA_Length"] = peptides_df["Epitope_Seq"].str.len()
/tmp/ipykernel_3702272/1950331011.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  peptides_df["Epitope_ID"] = peptides_df["IEDB_ID_URL"].str.split("/").str[-1]
/tmp/ipykernel_3702272/1950331011.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

(174, 6)

In [41]:
len(Panda24_All_PosEpitope_Seqs)

174

In [42]:
Panda24_IEDB_Pos_Epitope_DF["Epitope_Seq"].nunique()

174

In [43]:
Panda24_IEDB_Pos_Epitope_DF.head()

,Epitope_ID,Epitope_Seq,AA_Length,Molecule_Parent,Source_Molecule,UniProt_ID
0,216194,NVPSYRVSQYDIVDV,15,30S ribosomal protein S4,30S ribosomal protein S4,P9WH35
1,42794,MTEQQWNFAGIEAAA,15,6 kDa early secretory antigenic target,early secreted antigenic target 6 kDa,P9WNK7
2,106544,ISEAGQAMASTEGNV,15,6 kDa early secretory antigenic target,6 KDA EARLY SECRETORY ANTIGENIC TARGET ESXA (E...,P9WNK7
3,106684,NLARTISEAGQAMAS,15,6 kDa early secretory antigenic target,6 KDA EARLY SECRETORY ANTIGENIC TARGET ESXA (E...,P9WNK7
4,106686,NNALQNLARTISEAG,15,6 kDa early secretory antigenic target,6 KDA EARLY SECRETORY ANTIGENIC TARGET ESXA (E...,P9WNK7


### Panda-2024 - parse all ASSAYED peptides (N = 18626)

In [44]:
Panda24_IEDB_AllPeptides_DF = parse_IEDB_peptides_XLSX(Panda2024_IEDB_AllAssayedPeptides_XLSX)
Panda24_IEDB_AllPeptides_DF["PosEpi_InPanda24"] = Panda24_IEDB_AllPeptides_DF["Epitope_Seq"].isin(Panda24_All_PosEpitope_Seqs)

Panda24_IEDB_AllPeptides_SeqList = list(Panda24_IEDB_AllPeptides_DF["Epitope_Seq"].unique())

Panda24_IEDB_AllPeptides_DF.shape

/tmp/ipykernel_3702272/1950331011.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  peptides_df["AA_Length"] = peptides_df["Epitope_Seq"].str.len()
/tmp/ipykernel_3702272/1950331011.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  peptides_df["Epitope_ID"] = peptides_df["IEDB_ID_URL"].str.split("/").str[-1]
/tmp/ipykernel_3702272/1950331011.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

(18626, 7)

In [45]:
Panda24_IEDB_AllPeptides_DF["PosEpi_InPanda24"].value_counts()

PosEpi_InPanda24
False    18452
True       174
Name: count, dtype: int64

In [46]:
Panda24_IEDB_AllPeptides_DF["Epitope_Seq"].nunique()

18626

In [47]:
Panda24_IEDB_AllPeptides_DF.head()

,Epitope_ID,Epitope_Seq,AA_Length,Molecule_Parent,Source_Molecule,UniProt_ID,PosEpi_InPanda24
0,93,AADMWGPSSDPAWER,15,Diacylglycerol acyltransferase/mycolyltransfer...,Diacylglycerol acyltransferase/mycolyltransfer...,P9WQP1,False
1,153,AAGGHNAVFNFPPNG,15,Diacylglycerol acyltransferase/mycolyltransfer...,Diacylglycerol acyltransferase/mycolyltransfer...,P9WQP1,False
2,187,AAGTAAQAAVVRFQE,15,ESAT-6-like protein EsxB,10 kDa culture filtrate antigen EsxB,P9WNK5,True
3,222,AAIGLSMAGSSAMIL,15,Diacylglycerol acyltransferase/mycolyltransfer...,Diacylglycerol acyltransferase/mycolyltransfer...,P9WQP1,False
4,326,AANKQKQELDEISTN,15,ESAT-6-like protein EsxB,ESAT-6-like protein EsxB,P9WNK5,False


In [48]:
len(Lind16_IEDB_AllPeptides_SeqList)

761

In [49]:
len(Panda24_IEDB_AllPeptides_SeqList)

18626

# Combine and process all epitopes from both Lindestam-2016 & Panda-2024 (IEDB Submissions)

In [50]:
print(len(Lind16_IEDB_AllPeptides_SeqList))
print(len(Lind16_IEDB_PosEpitopes_SeqList))
print(len(Panda24_IEDB_AllPeptides_SeqList))
print(len(Panda24_IEDB_PosEpitopes_SeqList))


761
235
18626
174


In [51]:
print(len(set(Lind16_IEDB_AllPeptides_SeqList)))
print(len(set(Lind16_IEDB_PosEpitopes_SeqList)))
print(len(set(Panda24_IEDB_AllPeptides_SeqList)))
print(len(set(Panda24_IEDB_PosEpitopes_SeqList)))


761
235
18626
174


In [52]:
Lind16_Panda24_AllTestedPeptides = list(set(list(Lind16_IEDB_AllPeptides_SeqList) + list(Panda24_IEDB_AllPeptides_SeqList)))
len(Lind16_Panda24_AllTestedPeptides)


18733

In [53]:
LPM_AllPeptides_DF = pd.DataFrame(Lind16_Panda24_AllTestedPeptides)
LPM_AllPeptides_DF.columns = ["Epitope_Seq"]

LPM_AllPeptides_DF["Epitope_ID"] = "UnqPeptide_" + (LPM_AllPeptides_DF.index + 1).astype(str)

LPM_AllPeptides_DF["Assayed_Panda24"] = LPM_AllPeptides_DF["Epitope_Seq"].isin(Panda24_IEDB_AllPeptides_SeqList)
LPM_AllPeptides_DF["PosEpitope_Panda24"] = LPM_AllPeptides_DF["Epitope_Seq"].isin(Panda24_IEDB_PosEpitopes_SeqList)

LPM_AllPeptides_DF["Assayed_Lindestam16"]  = LPM_AllPeptides_DF["Epitope_Seq"].isin(Lind16_IEDB_AllPeptides_SeqList)
LPM_AllPeptides_DF["PosEpitope_Lindestam16"]  = LPM_AllPeptides_DF["Epitope_Seq"].isin(Lind16_IEDB_PosEpitopes_SeqList)

LPM_AllPeptides_DF.shape

(18733, 6)

In [54]:
LPM_AllPeptides_DF["Epitope_Seq"].nunique()

18733

In [55]:
LPM_AllPeptides_DF[["Assayed_Panda24", "Assayed_Lindestam16"]].value_counts()

Assayed_Panda24  Assayed_Lindestam16
True             False                  17972
                 True                     654
False            True                     107
Name: count, dtype: int64

In [56]:
LPM_AllPeptides_DF.query("Assayed_Lindestam16 == True & Assayed_Panda24 == True")[["PosEpitope_Panda24", "PosEpitope_Lindestam16"]].value_counts()

PosEpitope_Panda24  PosEpitope_Lindestam16
False               False                     461
                    True                      126
True                True                       57
                    False                      10
Name: count, dtype: int64

In [57]:
LPM_AllPeptides_DF.query("PosEpitope_Panda24 == True & PosEpitope_Lindestam16 == True").shape

(57, 6)

In [58]:
LPM_AllPeptides_DF.head(4)

,Epitope_Seq,Epitope_ID,Assayed_Panda24,PosEpitope_Panda24,Assayed_Lindestam16,PosEpitope_Lindestam16
0,FPTLNYAVSVAEACE,UnqPeptide_1,True,False,False,False
1,ERIPKFAHLPTVLGE,UnqPeptide_2,True,False,False,False
2,FPGVLVAARPVGMFR,UnqPeptide_3,True,False,False,False
3,GDPARTMRRMIGGLR,UnqPeptide_4,True,False,False,False


# Map all merged epitopes (`Panda2024` & `Lindestam2016`) to Rv ref proteins

In [59]:
LPM_AllPeptides_Mapped_DF = map_epitopes_to_proteins(LPM_AllPeptides_DF,
                                                     dictOf_H37Rv_ProtSeq_NCBI,
                                                     RvID_To_Symbol_Dict,
                                                     Symbol_To_RvID_Dict,
                                                     H37Rv_GenomeAnno_Genes_DF)

LPM_AllPeptides_Mapped_DF["Dataset"] = "Lindestam16_Panda24_Merged"

# Add info regarding whether peptide was assayed and if it was a positive epitope (For two datasets
LPM_AllPeptides_Mapped_DF["Assayed_Panda24"]     = LPM_AllPeptides_Mapped_DF["Epitope_Seq"].isin(Panda24_IEDB_AllPeptides_SeqList)
LPM_AllPeptides_Mapped_DF["PosEpitope_Panda24"]  = LPM_AllPeptides_Mapped_DF["Epitope_Seq"].isin(Panda24_IEDB_PosEpitopes_SeqList)

LPM_AllPeptides_Mapped_DF["Assayed_Lindestam16"]    = LPM_AllPeptides_Mapped_DF["Epitope_Seq"].isin(Lind16_IEDB_AllPeptides_SeqList)
LPM_AllPeptides_Mapped_DF["PosEpitope_Lindestam16"] = LPM_AllPeptides_Mapped_DF["Epitope_Seq"].isin(Lind16_IEDB_PosEpitopes_SeqList)

# Create a new column indicating whether a region has an epitope or not
LPM_AllPeptides_Mapped_DF['PosEpitope_Any'] = LPM_AllPeptides_Mapped_DF['PosEpitope_Panda24'] | LPM_AllPeptides_Mapped_DF['PosEpitope_Lindestam16'] 


LPM_AllPeptides_Mapped_DF["EpitopeSymbol_ID"] = LPM_AllPeptides_Mapped_DF["Epitope_ID"].astype(str) + "-" + LPM_AllPeptides_Mapped_DF["Symbol"] 

### Annotate all mapped peptides by whether they overlap with an HHR or not.
LPM_AllPeptides_Mapped_DF = bf.count_overlaps(LPM_AllPeptides_Mapped_DF, HM_MergedRegions_Anno_DF,
                                              cols1 = Epitope_CoordCols,
                                              cols2 = HmRegion_CoordCols)

LPM_AllPeptides_Mapped_DF.rename(columns={'count': 'N_HmRegion'}, inplace=True)

LPM_AllPeptides_Mapped_DF["HasHmRegion"] = LPM_AllPeptides_Mapped_DF["N_HmRegion"] > 0 

LPM_AllPeptides_Mapped_DF.shape

18733it [01:50, 169.78it/s]


(18741, 20)

In [60]:
LPM_AllPeptides_Mapped_DF["Assayed_Panda24"].value_counts()

Assayed_Panda24
True     18695
False       46
Name: count, dtype: int64

In [61]:
LPM_AllPeptides_Mapped_DF["PosEpitope_Panda24"].value_counts()

PosEpitope_Panda24
False    18575
True       166
Name: count, dtype: int64

In [62]:
LPM_AllPeptides_Mapped_DF["Assayed_Lindestam16"].value_counts()

Assayed_Lindestam16
False    17848
True       893
Name: count, dtype: int64

In [63]:
LPM_AllPeptides_Mapped_DF["PosEpitope_Lindestam16"].value_counts()

PosEpitope_Lindestam16
False    18423
True       318
Name: count, dtype: int64

In [64]:
LPM_AllPeptides_Mapped_DF.head()

,Epitope_ID,Epitope_Seq,Epitope_Len,RvID,Symbol,AA_Start,AA_End,Chrom,Rv_Start,Rv_End,EpitopeSeqFreqInAntigen,Dataset,Assayed_Panda24,PosEpitope_Panda24,Assayed_Lindestam16,PosEpitope_Lindestam16,PosEpitope_Any,EpitopeSymbol_ID,N_HmRegion,HasHmRegion
0,UnqPeptide_1,FPTLNYAVSVAEACE,15,Rv0322,udgA,368,383,NC_000962.3,390363,390408,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_1-udgA,0,False
1,UnqPeptide_2,ERIPKFAHLPTVLGE,15,Rv2992c,gltS,238,253,NC_000962.3,3349518,3349563,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_2-gltS,0,False
2,UnqPeptide_3,FPGVLVAARPVGMFR,15,Rv3628,ppa,66,81,NC_000962.3,4067620,4067665,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_3-ppa,0,False
3,UnqPeptide_4,GDPARTMRRMIGGLR,15,Rv3617,ephA,167,182,NC_000962.3,4058233,4058278,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_4-ephA,0,False
4,UnqPeptide_5,LLPYSYDDLDFNALL,15,Rv1282c,oppC,40,55,NC_000962.3,1435978,1436023,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_5-oppC,0,False


In [65]:
LPM_AllPeptides_Mapped_DF.query("EpitopeSeqFreqInAntigen > 1")

,Epitope_ID,Epitope_Seq,Epitope_Len,RvID,Symbol,AA_Start,AA_End,Chrom,Rv_Start,Rv_End,EpitopeSeqFreqInAntigen,Dataset,Assayed_Panda24,PosEpitope_Panda24,Assayed_Lindestam16,PosEpitope_Lindestam16,PosEpitope_Any,EpitopeSymbol_ID,N_HmRegion,HasHmRegion
2295,UnqPeptide_2295,DGNETNNPAPVSRVS,15,Rv3281,accE5,49,64,NC_000962.3,3663835,3663880,2,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_2295-accE5,0,False
2836,UnqPeptide_2834,AGFGAGVPGDGGIGG,15,Rv3508,PE_PGRS54,1295,1310,NC_000962.3,3934889,3934934,3,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_2834-PE_PGRS54,1,True
4105,UnqPeptide_4099,DGGDPLRPASPRLRS,15,Rv2859c,Rv2859c,8,23,NC_000962.3,3171577,3171622,3,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_4099-Rv2859c,1,True
4766,UnqPeptide_4755,LRSKVDAAWHLHELT,15,Rv2048c,pks12,1787,1802,NC_000962.3,2301580,2301625,2,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_4755-pks12,1,True
12482,UnqPeptide_12478,NTGAFISGNYSNGAF,15,Rv3343c,PPE54,1591,1606,NC_000962.3,3732117,3732162,2,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_12478-PPE54,1,True
13990,UnqPeptide_13984,ALANAGLSAAEVDVV,15,Rv2048c,pks12,320,335,NC_000962.3,2305981,2306026,2,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_13984-pks12,1,True
15108,UnqPeptide_15102,GDPLRPASPRLRSPV,15,Rv2859c,Rv2859c,10,25,NC_000962.3,3171571,3171616,2,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_15102-Rv2859c,1,True
15550,UnqPeptide_15552,AGGVGGAGGGTGGAG,15,Rv3508,PE_PGRS54,190,205,NC_000962.3,3931574,3931619,2,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_15552-PE_PGRS54,1,True
15551,UnqPeptide_15552,AGGVGGAGGGTGGAG,15,Rv3514,PE_PGRS57,190,205,NC_000962.3,3946363,3946408,2,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_15552-PE_PGRS57,1,True


### How many unique positive epitopes are detected? (307 unique epitope sequences across 134 unique proteins)

In [66]:
LPM_AllPeptides_Mapped_DF["PosEpitope_Any"].value_counts()

PosEpitope_Any
False    18317
True       424
Name: count, dtype: int64

In [67]:
LPM_AllPeptides_Mapped_DF.shape

(18741, 20)

In [68]:
LPM_PosEpi_Mapped_DF = LPM_AllPeptides_Mapped_DF.query("PosEpitope_Any == True")
LPM_PosEpi_Mapped_DF.shape

(424, 20)

In [69]:
LPM_PosEpi_Mapped_DF["Epitope_Seq"].nunique()

307

In [70]:
LPM_AllPeptides_Mapped_DF["Epitope_Seq"].nunique()

18371

In [71]:
LPM_AllPeptides_Mapped_DF["Symbol"].nunique()

3841

In [72]:
LPM_AllPeptides_Mapped_DF.query("Symbol == 'esxW'").shape

(19, 20)

In [73]:
LPM_AllPeptides_Mapped_DF.query("Symbol == 'esxM'").shape

(15, 20)

In [74]:
LPM_AllPeptides_Mapped_DF.query("Symbol == 'esxM'")["Epitope_Seq"].nunique()

15

In [75]:
#LPM_AllPeptides_Mapped_DF.query("Symbol == 'esxM'")

# Create summary of epitope mapping stats per gene (+ classify "High Confidence Antigens")

In [76]:
H37Rv_GenomeAnno_Genes_DF.head(3)

,Chrom,Start,End,Strand,H37rv_GeneID,Symbol,Feature,Functional_Category,Is_Pseudogene,Product,PEandPPE_Subfamily,ExcludedGroup_Category,Gene_Cat_V2
0,NC_000962.3,0,1524,+,Rv0001,dnaA,CDS,information pathways,No,Chromosomal replication initiator protein DnaA,NaN,NotExcluded,information pathways
1,NC_000962.3,2051,3260,+,Rv0002,dnaN,CDS,information pathways,No,DNA polymerase III (beta chain) DnaN (DNA nucl...,NaN,NotExcluded,information pathways
2,NC_000962.3,3279,4437,+,Rv0003,recF,CDS,information pathways,No,DNA replication and repair protein RecF (singl...,NaN,NotExcluded,information pathways


## A) Create a summary DF that epitope mapping results quantified per gene/protein

In [77]:
LPM_AllPeptides_Mapped_DF.shape

(18741, 20)

In [78]:
# Count the number of positive and negative epitopes per gene
LPM_epitope_counts_per_gene = LPM_AllPeptides_Mapped_DF.groupby(['Symbol', 'PosEpitope_Any']).size().unstack(fill_value=0)

# Sort the DataFrame by the total number of epitopes tested per gene
LPM_epitope_counts_per_gene['Total'] = LPM_epitope_counts_per_gene.sum(axis=1)
LPM_epitope_counts_per_gene = LPM_epitope_counts_per_gene[[True, False, "Total"]]
LPM_epitope_counts_per_gene.columns = ["N_Pos", "N_Neg", "Total"]

LPM_epitope_counts_per_gene['Positive_Proportion'] = LPM_epitope_counts_per_gene["N_Pos"] / LPM_epitope_counts_per_gene['Total']

LPM_epitope_counts_per_gene_sorted = LPM_epitope_counts_per_gene.sort_values('Total', ascending=False)
LPM_epitope_counts_per_gene_sorted.shape

(3841, 4)

In [79]:
LPM_epitope_counts_per_gene_sorted.head(1)

,N_Pos,N_Neg,Total,Positive_Proportion
Symbol,,,,
PPE42,16,106,122,0.131148


In [80]:
LPM_epitope_counts_per_gene_sorted.query("N_Pos >= 2").shape

(53, 4)

In [81]:
LPM_epitope_counts_per_gene_sorted.query("N_Pos >= 2")["N_Pos"].sum()

342

## B) Create a summary DF of all H37Rv genes with epitope mapping summary stats

In [89]:
Rv_GeneCols_Trim = ["Chrom", "Start", "End", "Strand", "Feature", "H37rv_GeneID", "Symbol", "Functional_Category", "Gene_Cat_V2" ]

Rv_Genes_EpitopeSummary_DF = H37Rv_GenomeAnno_Genes_DF[Rv_GeneCols_Trim]

Rv_Genes_EpitopeSummary_DF = pd.merge(Rv_Genes_EpitopeSummary_DF,
                                      LPM_epitope_counts_per_gene_sorted,
                                      on = "Symbol")

Rv_Genes_EpitopeSummary_DF["Middle"] = (Rv_Genes_EpitopeSummary_DF["End"] + Rv_Genes_EpitopeSummary_DF["Start"] ) / 2
Rv_Genes_EpitopeSummary_DF["Length"] = Rv_Genes_EpitopeSummary_DF["End"] - Rv_Genes_EpitopeSummary_DF["Start"] # The start and end coords are 0-based, so no "+ 1" at the end

Rv_Genes_EpitopeSummary_DF["AnyPosEpitope"] = Rv_Genes_EpitopeSummary_DF["N_Pos"] > 0

Rv_Genes_EpitopeSummary_DF["Antigen_LVL2"] = Rv_Genes_EpitopeSummary_DF["N_Pos"] >= 2


Rv_Genes_EpitopeSummary_DF = bf.count_overlaps(Rv_Genes_EpitopeSummary_DF, HM_MergedRegions_Anno_DF,
                                               cols1 = GenomeAnno_CoordCols,
                                               cols2 = HmRegion_CoordCols).rename(columns={'count': 'N_HmRegion'}) 

Rv_Genes_EpitopeSummary_DF["HasHmRegion"] =  Rv_Genes_EpitopeSummary_DF["N_HmRegion"] > 0

Rv_Genes_EpitopeSummary_DF["AntigenLVL2_And_HHR_Comb"] = Rv_Genes_EpitopeSummary_DF["Antigen_LVL2"].astype(str).replace("True", "Antigen").replace("False", "NonReactive") + "-" + Rv_Genes_EpitopeSummary_DF["HasHmRegion"].astype(str).replace("True", "HHR").replace("False", "Unq")    


In [90]:
Rv_Genes_EpitopeSummary_DF["Antigen_LVL2"].value_counts()

Antigen_LVL2
False    3788
True       53
Name: count, dtype: int64

In [91]:
Rv_Genes_EpitopeSummary_DF["AntigenLVL2_And_HHR_Comb"].value_counts()

AntigenLVL2_And_HHR_Comb
NonReactive-Unq    3584
NonReactive-HHR     204
Antigen-Unq          33
Antigen-HHR          20
Name: count, dtype: int64

In [92]:
Rv_Genes_EpitopeSummary_DF.head(3)

,Chrom,Start,End,Strand,Feature,H37rv_GeneID,Symbol,Functional_Category,Gene_Cat_V2,N_Pos,N_Neg,Total,Positive_Proportion,Middle,Length,AnyPosEpitope,Antigen_LVL2,N_HmRegion,HasHmRegion,AntigenLVL2_And_HHR_Comb
0,NC_000962.3,0,1524,+,CDS,Rv0001,dnaA,information pathways,information pathways,0,8,8,0.0,762.0,1524,False,False,0,False,NonReactive-Unq
1,NC_000962.3,2051,3260,+,CDS,Rv0002,dnaN,information pathways,information pathways,0,6,6,0.0,2655.5,1209,False,False,0,False,NonReactive-Unq
2,NC_000962.3,3279,4437,+,CDS,Rv0003,recF,information pathways,information pathways,0,6,6,0.0,3858.0,1158,False,False,0,False,NonReactive-Unq


## C) Output epitope stats per gene  to TSV

In [93]:
Repo_Epitope_MainDir = "../../Data/220813_MtbEpitopes"

Rv_Genes_Epitope_SummStats_TSV = f"{Repo_Epitope_MainDir}/240820.RvGene.EpitopeMappingStats.V1.tsv"

Rv_Genes_EpitopeSummary_DF.to_csv(Rv_Genes_Epitope_SummStats_TSV, sep="\t", index=False)

In [94]:
!wc -l $Rv_Genes_Epitope_SummStats_TSV

3842 ../../Data/220813_MtbEpitopes/240820.RvGene.EpitopeMappingStats.V1.tsv


# Add "Antigen-LVL2" classification info to epitope mapping results DF

### Identify antigens which has **2 or more** positive epitopes

In [95]:
Antigens_LVL2 = list(Rv_Genes_EpitopeSummary_DF.query("N_Pos >= 2")["Symbol"].unique())
len(Antigens_LVL2)

53

## Add "High Confidence Antigen" classification info as column

In [96]:
LPM_AllPeptides_Mapped_DF["Antigen_LVL2"] = LPM_AllPeptides_Mapped_DF["Symbol"].isin(Antigens_LVL2) 
LPM_AllPeptides_Mapped_DF.shape

(18741, 21)

In [97]:
LPM_AllPeptides_Mapped_DF.query("Antigen_LVL2 == True")["Symbol"].nunique()

53

# Output epitope mapping info for entire dataset (to TSV)

In [98]:
Repo_Epitope_MainDir = "../../Data/220813_MtbEpitopes"


Lind16_AllAssayedPeptides_Mapped_TSV = f"{Repo_Epitope_MainDir}/240815.Lindestram2016.PeptidesMappedToRv.AllAssayed.V1.tsv" 
Lind16_Peptides_HLAInfo_TSV = f"{Repo_Epitope_MainDir}/240815.Lindestram2016.HLA_ResponseInfo.V1.tsv" 

LPM_AllAssayedPeptides_Mapped_TSV = f"{Repo_Epitope_MainDir}/240820.Panda24_Lind16.Merged.PeptidesMappedToRv.AllAssayed.V1.tsv" 


# Output TSVs for each dataset

Lind16_IEDB_AllPeptides_HLAInfo_DF.to_csv(Lind16_Peptides_HLAInfo_TSV, sep = "\t", index=False)

LPM_AllPeptides_Mapped_DF.to_csv(LPM_AllAssayedPeptides_Mapped_TSV, sep = "\t", index=False)


In [99]:
!wc -l $Lind16_Peptides_HLAInfo_TSV

1026 ../../Data/220813_MtbEpitopes/240815.Lindestram2016.HLA_ResponseInfo.V1.tsv


In [100]:
!wc -l $LPM_AllAssayedPeptides_Mapped_TSV

18742 ../../Data/220813_MtbEpitopes/240820.Panda24_Lind16.Merged.PeptidesMappedToRv.AllAssayed.V1.tsv


## Extra exploration

In [101]:
Rv_Genes_EpitopeSummary_DF["Feature"].value_counts()

Feature
CDS    3841
Name: count, dtype: int64

In [102]:
Antigens_LVL2 = list(Rv_Genes_EpitopeSummary_DF.query("N_Pos >= 2")["Symbol"].unique())
len(Antigens_LVL2)

53

In [103]:
Rv_Genes_EpitopeSummary_DF["Antigen_LVL2"].value_counts()

Antigen_LVL2
False    3788
True       53
Name: count, dtype: int64

In [104]:
Rv_Genes_EpitopeSummary_DF["AntigenLVL2_And_HHR_Comb"].value_counts()

AntigenLVL2_And_HHR_Comb
NonReactive-Unq    3584
NonReactive-HHR     204
Antigen-Unq          33
Antigen-HHR          20
Name: count, dtype: int64

In [105]:
Rv_Genes_EpitopeSummary_DF.query("AntigenLVL2_And_HHR_Comb == 'Antigen-HHR' ")["N_Pos"].sum()

167

In [106]:
#Rv_Genes_EpitopeSummary_DF.query("AntigenLVL2_And_HHR_Comb == 'Antigen-HHR' ")

#### Extra exploration

In [107]:
Rv_Genes_EpitopeSummary_DF["Antigen_LVL2"].value_counts()

Antigen_LVL2
False    3788
True       53
Name: count, dtype: int64

In [108]:
Rv_Genes_EpitopeSummary_DF.query("N_Pos >= 1").shape

(135, 20)

In [109]:
Rv_Genes_EpitopeSummary_DF.query("N_Pos >= 2").shape

(53, 20)

In [110]:
Rv_Genes_EpitopeSummary_DF.query("N_Pos >= 3").shape

(31, 20)

In [111]:
Rv_Genes_EpitopeSummary_DF.query("N_Pos >= 2")["N_Pos"].sum()

342